<a href="https://colab.research.google.com/github/Atharva-AAS/Atharva-AAS/blob/main/Olist_Data_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Olist E-Commerce Analytics: Data Profiling & Core Business Metrics***

## 1. Project Overview

The objective of this project is to analyze sales performance, customer behavior, and delivery logistics for Olist, a Brazilian e-commerce platform. This notebook represents Phase 1 of the analytics workflow: extracting raw relational data, verifying data integrity (primary keys and missing values), handling duplicates, and engineering core business delivery metrics.

### 2. Dataset / Business Context

The Olist dataset represents approximately 100,000 real, anonymized commercial orders from 2016 to 2018. The data is structured as a relational database spread across multiple CSV files (Orders, Items, Customers, Payments, Reviews). Careful data profiling is required before analysis to prevent common business intelligence errors, such as double-counting revenue due to one-to-many relationships between orders, items, and payments.


## 3. Imports & Setup

In [46]:
import os

print("Searching your Google Drive for the dataset...")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "olist_orders_dataset.csv" in files:
        print("\n✅ FOUND IT! Copy and paste this exact line into Cell 2:")
        print(f'base_path = "{root}"')
        break
else:
    print("\n❌ The file is NOT in your Google Drive. You may not have uploaded it.")


Searching your Google Drive for the dataset...

✅ FOUND IT! Copy and paste this exact line into Cell 2:
base_path = "/content/drive/MyDrive/Olist_Dataset"


In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:

import pandas as pd
import os

# Define the dataset directory (Update this path if running locally)
base_path = "/content/drive/MyDrive/Olist_Dataset"




## 4. Data Loading & Profiling
Before joining any tables, we must verify our assumptions about Primary Keys and missing values. If we blindly join tables where primary keys are duplicated, we risk artificially inflating business metrics like Total Revenue.



In [49]:
# Define the core files and the columns we EXPECT to be unique (Primary Keys)
files_to_check = {
    'olist_orders_dataset.csv': 'order_id',
    'olist_order_items_dataset.csv': ['order_id', 'order_item_id'],
    'olist_customers_dataset.csv': 'customer_id',
    'olist_order_payments_dataset.csv': ['order_id', 'payment_sequential'],
    'olist_order_reviews_dataset.csv': 'review_id',
    'olist_products_dataset.csv': 'product_id',
    'olist_sellers_dataset.csv': 'seller_id'
}

print("--- OLIST DATASET VERIFICATION ---\n")

for file, pk in files_to_check.items():
    file_path = os.path.join(base_path, file)

    try:
        df = pd.read_csv(file_path)
        print(f"📁 File: {file}")
        print(f"Row Count: {len(df):,}")

        # Null check
        null_counts = df.isnull().sum()
        columns_with_nulls = null_counts[null_counts > 0]

        if not columns_with_nulls.empty:
            print("Null Values Found:")
            for col, count in columns_with_nulls.items():
                print(f"  - {col}: {count:,} missing ({(count / len(df)) * 100:.2f}%)")
        else:
            print("Null Values: None")

        # Primary key check
        if isinstance(pk, list):
            duplicate_count = df.duplicated(subset=pk).sum()
            print(f"Primary Key Check ({'+'.join(pk)}): {'UNIQUE ✅' if duplicate_count == 0 else 'DUPLICATES ❌'}\n")
        else:
            print(f"Primary Key Check ({pk}): {'UNIQUE ✅' if df[pk].is_unique else 'DUPLICATES ❌'}\n")

    except FileNotFoundError:
        print(f"❌ Error: {file} not found at {file_path}\n")


--- OLIST DATASET VERIFICATION ---

📁 File: olist_orders_dataset.csv
Row Count: 99,441
Null Values Found:
  - order_approved_at: 160 missing (0.16%)
  - order_delivered_carrier_date: 1,783 missing (1.79%)
  - order_delivered_customer_date: 2,965 missing (2.98%)
Primary Key Check (order_id): UNIQUE ✅

📁 File: olist_order_items_dataset.csv
Row Count: 112,650
Null Values: None
Primary Key Check (order_id+order_item_id): UNIQUE ✅

📁 File: olist_customers_dataset.csv
Row Count: 99,441
Null Values: None
Primary Key Check (customer_id): UNIQUE ✅

📁 File: olist_order_payments_dataset.csv
Row Count: 103,886
Null Values: None
Primary Key Check (order_id+payment_sequential): UNIQUE ✅

📁 File: olist_order_reviews_dataset.csv
Row Count: 99,224
Null Values Found:
  - review_comment_title: 87,656 missing (88.34%)
  - review_comment_message: 58,247 missing (58.70%)
Primary Key Check (review_id): DUPLICATES ❌

📁 File: olist_products_dataset.csv
Row Count: 32,951
Null Values Found:
  - product_category_

## 5. Exploratory Data Analysis: Investigating Anomalies
The profiling script revealed an anomaly: `review_id` is NOT unique. We need to investigate this before trusting the Reviews table. We will also inspect the distribution of `order_status` in the core Orders table.


In [50]:
# Investigate duplicated reviews
reviews_path = os.path.join(base_path, "olist_order_reviews_dataset.csv")
reviews = pd.read_csv(reviews_path)

duplicate_reviews = reviews[reviews.duplicated("review_id", keep=False)].sort_values("review_id")
print(f"Total duplicated review rows: {len(duplicate_reviews)}")

# Load core orders table and check order status
orders_path = os.path.join(base_path, "olist_orders_dataset.csv")
orders = pd.read_csv(orders_path)

print("\nDistribution of Order Status:")
print(orders['order_status'].value_counts())


Total duplicated review rows: 1603

Distribution of Order Status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 6. Data Cleaning & Preparation
The `olist_orders_dataset` contains multiple timestamp columns stored as strings. We need to convert these to datetime objects to allow for time-series analysis and delivery performance calculations.


In [51]:
# Convert all timestamp columns to datetime objects
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Verify data types are now correctly set to datetime64[ns]
print(orders.dtypes)


order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


## 7. Business/Customer/Order Insights (Feature Engineering)
To answer business questions regarding shipping performance, we will engineer two new metrics:
1. **Delivery Days**: Actual days taken from purchase to delivery.
2. **Delivery Delay Days**: The difference between the actual delivery date and the estimated delivery date (positive values indicate a late delivery).


In [52]:
# Calculate delivery duration and delays
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

# Create a clean, focused Fact Table for downstream analysis
fact_orders = orders[[
    'order_id',
    'customer_id',
    'order_status',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'delivery_days',
    'delivery_delay_days'
]].copy()

# Inspect the top 10 longest deliveries
print("Top 10 Longest Deliveries:")
display(fact_orders.sort_values(by='delivery_days', ascending=False).head(10))


Top 10 Longest Deliveries:


,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
19590,ca07593549f1816d26a572e06dc1eab6,75683a92331068e2d281b11a7866ba44,delivered,2017-02-21 23:31:27,2017-09-19 14:36:39,2017-03-22,209.0,181.0
55619,1b3190b2dfa9d789e1f14c05b647a14a,d306426abe5fca15e54b645e4462dc7b,delivered,2018-02-23 14:57:35,2018-09-19 23:24:07,2018-03-15,208.0,188.0
61610,440d0d17af552815d15a9e41abe49359,7815125148cfa1e8c7fee1ff7974f16c,delivered,2017-03-07 23:59:51,2017-09-19 15:12:50,2017-04-07,195.0,165.0
70307,2fb597c2f772eca01b1f5c561bf6cc7b,217906bc11a32c1e470eb7e08584894b,delivered,2017-03-08 18:09:02,2017-09-19 14:33:17,2017-04-17,194.0,155.0
38509,0f4519c5f1c541ddec9f21b3bddd533a,1a8a4a30dc296976717f44e7801fdeef,delivered,2017-03-09 13:26:57,2017-09-19 14:38:21,2017-04-11,194.0,161.0
89130,285ab9426d6982034523a855f55a885e,9cf2c3fa2632cee748e1a59ca9d09b21,delivered,2017-03-08 22:47:40,2017-09-19 14:00:04,2017-04-06,194.0,166.0
11399,47b40429ed8cce3aee9199792275433f,cb2caaaead400c97350c37a3fc536867,delivered,2018-01-03 09:44:01,2018-07-13 20:51:31,2018-01-19,191.0,175.0
81401,2fe324febf907e3ea3f2aa9650869fa5,65b14237885b3972ebec28c0f7dd2220,delivered,2017-03-13 20:17:10,2017-09-19 17:00:07,2017-04-05,189.0,167.0
54480,2d7561026d542c8dbd8f0daeadf67a43,8199345f57c6d1cbe9701f92481beb8d,delivered,2017-03-15 11:24:27,2017-09-19 14:38:18,2017-04-13,188.0,159.0
68769,c27815f7e3dd0b926b58552628481575,f85e9ec0719b16dc4dd0edd438793553,delivered,2017-03-15 23:23:17,2017-09-19 17:14:25,2017-04-10,187.0,162.0


### 4.1 Data Integrity Review
We have already verified the primary keys and null values in the previous sections. The data is loaded and ready for analysis.

In [53]:
import pandas as pd
import os

# Using the base_path defined in cell Xau07RuxHy2y to avoid FileNotFoundError
reviews_path = os.path.join(base_path, "olist_order_reviews_dataset.csv")

reviews = pd.read_csv(reviews_path)

# Identify duplicate reviews based on review_id
duplicate_reviews = reviews[
    reviews.duplicated("review_id", keep=False)
].sort_values("review_id")

duplicate_reviews.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [54]:
### 5.1 Order Status Distribution Summary

In [55]:
### 6.1 Timestamp Verification

In [56]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]
delivery_days,float64
delivery_delay_days,float64


In [57]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days


orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

In [58]:
orders[
    [
        'order_id',
        'order_status',
        'delivery_days',
        'delivery_delay_days'
    ]
].head(10)

,order_id,order_status,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.0,-10.0
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,16.0,-6.0
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,NaN,NaN
7,6514b8ad8028c9f2cc2374ded245783f,delivered,9.0,-12.0
8,76c6e866289321a7c93b82b54852dc33,delivered,9.0,-32.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,18.0,-7.0


In [59]:
fact_orders = orders[
    [
        'order_id',
        'customer_id',
        'order_status',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date',
        'delivery_days',
        'delivery_delay_days'
    ]
].copy()

fact_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,-10.0


In [60]:
fact_orders.shape

(99441, 8)

In [61]:
fact_orders.describe()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days
count,99441,96476,99441,96476.000000,96476.000000
mean,2017-12-31 08:43:12.776581120,2018-01-14 12:09:19.035542272,2018-01-24 03:08:37.730111232,12.094086,-11.876881
min,2016-09-04 21:15:19,2016-10-11 13:46:32,2016-09-30 00:00:00,0.000000,-147.000000
25%,2017-09-12 14:46:19,2017-09-25 22:07:22.249999872,2017-10-03 00:00:00,6.000000,-17.000000
50%,2018-01-18 23:04:36,2018-02-02 19:28:10.500000,2018-02-15 00:00:00,10.000000,-12.000000
75%,2018-05-04 15:42:16,2018-05-15 22:48:52.249999872,2018-05-25 00:00:00,15.000000,-7.000000
max,2018-10-17 17:30:18,2018-10-17 13:22:46,2018-11-12 00:00:00,209.000000,188.000000
std,NaN,NaN,NaN,9.551746,10.183854


In [62]:
fact_orders.sort_values(
    by='delivery_days',
    ascending=False
)[
    [
        'order_id',
        'order_status',
        'delivery_days',
        'delivery_delay_days'
    ]
].head(10)

,order_id,order_status,delivery_days,delivery_delay_days
19590,ca07593549f1816d26a572e06dc1eab6,delivered,209.0,181.0
55619,1b3190b2dfa9d789e1f14c05b647a14a,delivered,208.0,188.0
61610,440d0d17af552815d15a9e41abe49359,delivered,195.0,165.0
70307,2fb597c2f772eca01b1f5c561bf6cc7b,delivered,194.0,155.0
38509,0f4519c5f1c541ddec9f21b3bddd533a,delivered,194.0,161.0
89130,285ab9426d6982034523a855f55a885e,delivered,194.0,166.0
11399,47b40429ed8cce3aee9199792275433f,delivered,191.0,175.0
81401,2fe324febf907e3ea3f2aa9650869fa5,delivered,189.0,167.0
54480,2d7561026d542c8dbd8f0daeadf67a43,delivered,188.0,159.0
68769,c27815f7e3dd0b926b58552628481575,delivered,187.0,162.0


In [63]:
items_path = "/content/drive/MyDrive/Olist_Dataset/olist_order_items_dataset.csv"
payments_path = "/content/drive/MyDrive/Olist_Dataset/olist_order_payments_dataset.csv"

order_items = pd.read_csv(items_path)
payments = pd.read_csv(payments_path)

print(order_items.shape)
print(payments.shape)

(112650, 7)
(103886, 5)


In [64]:
# Check for logical anomalies in Order Items (Prices and Freight)
print("--- ORDER ITEMS SUMMARY ---")
display(order_items[['price', 'freight_value']].describe())

# Check for anomalies in Payment Types
print("\n--- PAYMENT TYPES DISTRIBUTION ---")
print(payments['payment_type'].value_counts())

# Check for anomalies in Payment Values (Negative payments?)
print("\n--- PAYMENT VALUE SUMMARY ---")
display(payments[['payment_value']].describe())

--- ORDER ITEMS SUMMARY ---


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000



--- PAYMENT TYPES DISTRIBUTION ---
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

--- PAYMENT VALUE SUMMARY ---


,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000
